# CAT-1 — Same Iceberg table, two catalogs (Hadoop vs Nessie REST)

The lesson is **not** that "Iceberg on Nessie" is a different format — it's the *same* Iceberg table. What differs is **who owns the `current-snapshot` pointer**:

| Catalog | Where the pointer lives | Commit atomicity on S3 |
|---|---|---|
| **Hadoop** (filesystem) | a `version-hint.text` file in S3 that writers race to rename | rename on S3 is copy+delete → **not atomic** (the LAK-10 anti-pattern) |
| **Nessie** (REST) | inside Nessie's own state, served over HTTP | commit = atomic pointer advance in the service |

We create the *same* table on both, look at where each keeps its pointer (real files on S3, not a drawing), and prove that **Spark and Nessie's own REST API agree** on the current snapshot — the definition of "the catalog is the single source of truth."

> Prereqs: `make up` (Spark + MiniStack) **and** `make catalogs-up` (Nessie).

In [1]:
import json, urllib.parse, urllib.request
from common.spark_session import spark
from common.table_meta import s3_client   # thin S3 boilerplate (reused)

# Two tables — same schema, same data — one per catalog.
HADOOP_TABLE = "iceberg_catalog.default.cat1_orders"   # Hadoop / filesystem catalog
NESSIE_TABLE = "nessie_catalog.marts.cat1_orders"      # Nessie REST catalog
NESSIE_KEY   = "marts.cat1_orders"                      # dotted key in Nessie's REST path

# Idempotent reset.
spark.sql(f"DROP TABLE IF EXISTS {HADOOP_TABLE}")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie_catalog.marts")
spark.sql(f"DROP TABLE IF EXISTS {NESSIE_TABLE}")
print("reset done")

reset done


## 1. Identical DDL on both catalogs

The `CREATE TABLE` differs only in the catalog prefix of the name. A reader can't tell them apart — the catalog choice is transparent to the query.

In [2]:
spark.sql(f"CREATE TABLE {HADOOP_TABLE} (id BIGINT, amount DOUBLE) USING iceberg")
spark.sql(f"INSERT INTO {HADOOP_TABLE} VALUES (1, 50.0), (2, 75.0)")
print(f"Hadoop-catalog table: {HADOOP_TABLE}")
spark.sql(f"SELECT * FROM {HADOOP_TABLE} ORDER BY id").show()

Hadoop-catalog table: iceberg_catalog.default.cat1_orders
+---+------+
| id|amount|
+---+------+
|  1|  50.0|
|  2|  75.0|
+---+------+



In [3]:
spark.sql(f"CREATE TABLE {NESSIE_TABLE} (id BIGINT, amount DOUBLE) USING iceberg")
spark.sql(f"INSERT INTO {NESSIE_TABLE} VALUES (1, 50.0), (2, 75.0)")
print(f"Nessie-catalog table: {NESSIE_TABLE}")
spark.sql(f"SELECT * FROM {NESSIE_TABLE} ORDER BY id").show()

Nessie-catalog table: nessie_catalog.marts.cat1_orders
+---+------+
| id|amount|
+---+------+
|  1|  50.0|
|  2|  75.0|
+---+------+



## 2. Where does the "current snapshot" pointer live?

Both tables store identical Parquet under `data/` and identical `metadata.json` snapshots. The difference is the **pointer** to the *current* metadata:

- **Hadoop catalog** writes a `version-hint.text` file in S3. Two concurrent writers must atomically rename `version-hint.text.new → version-hint.text`. On HDFS that's atomic; on **S3 it's copy+delete → not atomic** → lost commits.
- **Nessie** keeps the pointer in its own state and serves it over HTTP — there is **no `version-hint.text`**.

Let's list the real files on S3 and confirm.

In [4]:
s3 = s3_client()

def list_prefix(prefix, label):
    print(f"\n{label}  (s3://warehouse/{prefix}):")
    keys = [o["Key"] for o in s3.list_objects_v2(Bucket="warehouse", Prefix=prefix).get("Contents", [])]
    for k in sorted(keys):
        mark = "   ← the current-pointer file!" if k.endswith("version-hint.text") else ""
        print(f"   {k[len(prefix):]}{mark}")
    print(f"   → version-hint.text present? {any(k.endswith('version-hint.text') for k in keys)}")

list_prefix("iceberg/default/cat1_orders/metadata/", "HADOOP catalog metadata")
list_prefix("nessie/marts/", "NESSIE catalog files")


HADOOP catalog metadata  (s3://warehouse/iceberg/default/cat1_orders/metadata/):
   09e6f04a-61dc-4059-b35e-bdc80422a401-m0.avro
   snap-349652590587500331-1-09e6f04a-61dc-4059-b35e-bdc80422a401.avro
   v1.metadata.json
   v2.metadata.json
   version-hint.text   ← the current-pointer file!
   → version-hint.text present? True

NESSIE catalog files  (s3://warehouse/nessie/marts/):
   cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/data/00000-3-c789630d-1c20-460e-b05a-e8e803e4ef8e-0-00001.parquet
   cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/data/00001-4-c789630d-1c20-460e-b05a-e8e803e4ef8e-0-00001.parquet
   cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/metadata/00000-a8e71fb0-e590-4d67-a6a7-4ace3c681c7d.metadata.json
   cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/metadata/00001-f3d2e21e-3b73-4a6f-b770-4c1b8d2459f3.metadata.json
   cat1_orders_855b3c1d-6dad-43df-9fb9-e68a572710e6/metadata/39a4cb96-c05f-4c90-816e-c030521e92a6-m0.avro
   cat1_orders_855b3c1d-6dad-43df-9fb9

## 3. The switch is one line of config — not a data rewrite

Everything under `data/` is the same Parquet either way. Moving to a REST catalog is a **config change**, already in `conf/spark-defaults.conf`:

```
spark.sql.catalog.nessie_catalog        org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.nessie_catalog.type   rest
spark.sql.catalog.nessie_catalog.uri    http://nessie:19120/iceberg/
```

**Reach for a REST catalog when:** object storage is the backend, more than one writer touches a table, or you need cross-team RBAC / branching (see CAT-2, CAT-3).

**The Hadoop catalog is fine when:** single-writer local prototypes, or when you're teaching the raw on-disk metadata layout (that's why LAK-1…10 keep it).

## 4. Prove it — Spark and Nessie's own API agree on the current snapshot

The honest test of "single source of truth": ask **two independent parties** — Spark (via the Iceberg REST catalog) and **Nessie's REST API directly** (raw HTTP) — for the current `snapshot_id`. They must match.

> (This is a *two-way* check. The `common.catalog_meta.nessie_health` helper wraps the exact same REST read — it isn't a third independent source, so we don't pretend it is.)

In [5]:
# Source A — Spark's answer, via the Iceberg REST catalog
spark_snap = spark.sql(
    f"SELECT snapshot_id FROM {NESSIE_TABLE}.snapshots ORDER BY committed_at DESC LIMIT 1"
).collect()[0]["snapshot_id"]

# Source B — Nessie's OWN REST API, read directly (GET /api/v2/trees/{ref}/contents/{key})
url = f"http://localhost:19120/api/v2/trees/main/contents/{urllib.parse.quote(NESSIE_KEY, safe='')}"
with urllib.request.urlopen(url, timeout=5) as r:
    content = (json.loads(r.read()).get("content") or {})
nessie_snap = content.get("snapshotId") or (content.get("metadata") or {}).get("current-snapshot-id")

print(f"  Spark  (.snapshots)      : {spark_snap}")
print(f"  Nessie (REST /contents)  : {nessie_snap}")
assert spark_snap == nessie_snap, f"mismatch: {spark_snap} vs {nessie_snap}"
print("\nAgreed — Spark and Nessie's own API report the same current snapshot.")
print("That equality is what 'the catalog is the single source of truth' means.")

  Spark  (.snapshots)      : 393379293653393052
  Nessie (REST /contents)  : 393379293653393052

Agreed — Spark and Nessie's own API report the same current snapshot.
That equality is what 'the catalog is the single source of truth' means.


In [6]:
spark.sql(f"DROP TABLE IF EXISTS {HADOOP_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {NESSIE_TABLE}")
print("dropped both tables — `make clean` clears .tmp/ for a fresh warehouse")

dropped both tables — `make clean` clears .tmp/ for a fresh warehouse


## What you just saw

- **Iceberg-on-Nessie is the same table format** — only the *pointer owner* changed.
- The Hadoop catalog's pointer is a **file on S3** (`version-hint.text`) that writers race to rename non-atomically; Nessie serves the pointer from a **service**, making commits atomic on object storage.
- Switching is **one config line**, not a data migration.
- "Single source of truth" is falsifiable: Spark and the catalog's own API must agree on the current snapshot.

### Try it yourself
- `INSERT` again into the Nessie table and re-run cell 4 — the snapshot advances in lock-step on both sides.
- Open the API playground ([`../catalog_api_playground.ipynb`](../catalog_api_playground.ipynb)) to poke Nessie's REST API by hand.

**Next:** [CAT-3 — Git-like branching on Nessie](./cat3_branching.ipynb) · [CAT-2 — Polaris RBAC](../cat2_polaris_rbac.ipynb)